In [1]:
import sys
print(sys.executable)

%pip install pandas
%pip show scikit-learn
%pip show lightgbm
%pip uninstall clip clip-by-openai -y
%pip install git+https://github.com/openai/CLIP.git
%pip install tqdm

import sklearn
print(sklearn.__version__)

import lightgbm
print(lightgbm.__version__)

import clip
print(hasattr(clip, 'load'))

/work/xingyuan-k/ks_pc/.venv/bin/python
Note: you may need to restart the kernel to use updated packages.
Name: scikit-learn
Version: 1.9.0
Summary: A set of python modules for machine learning and data mining
Home-page: 
Author: 
Author-email: 
License: 
Location: /work/xingyuan-k/ks_pc/.venv/lib/python3.12/site-packages
Requires: joblib, narwhals, numpy, scipy, threadpoolctl
Required-by: 
Note: you may need to restart the kernel to use updated packages.
Name: lightgbm
Version: 4.7.0
Summary: LightGBM Python-package
Home-page: 
Author: 
Author-email: 
License: 
Location: /work/xingyuan-k/ks_pc/.venv/lib/python3.12/site-packages
Requires: narwhals, numpy, scipy
Required-by: 
Note: you may need to restart the kernel to use updated packages.
Found existing installation: clip 1.0
Uninstalling clip-1.0:
  Successfully uninstalled clip-1.0
Note: you may need to restart the kernel to use updated packages.
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-libkh6ka
  Running c

In [2]:
# Build a clean master table from posting data
import pandas as pd

def load_posts(path, source):
    df = pd.read_csv(path, on_bad_lines='skip')
    df = df.rename(columns={'longitude': 'lng'})
    df['source'] = source
    # Drop personal info immediately (never leak email)
    df = df.drop(columns=[c for c in ['email'] if c in df.columns])
    return df

a = load_posts("../data/app_data/posting_data/QuestPOST_csv/Posts.csv", "quest")
b = load_posts("../data/app_data/posting_data/gnavi_db_csv/Posts.csv", "gnavi")
posts = pd.concat([a, b], ignore_index=True)

# Filenames encode lat_lng, we already have lat/lng columns -> keep numeric
posts = posts.dropna(subset=['lat', 'lng'])
# Keep only local image filename (strip the server URL / personal path)
posts['img_file'] = posts['image_path'].astype(str).str.split('/').str[-1]

In [3]:
# Take only the FIRST tag before comma, strip '#', keep top-K classes as "main label"
def clean_label(col, topk=6):
    s = posts[col].astype(str).str.replace('#','', regex=False)
    s = s.str.split(',').str[0].str.strip()       # first tag only
    top = s.value_counts().index[:topk]           # keep frequent classes
    return s.where(s.isin(top), other='その他')    # rest -> "other"

posts['Y1_mood']  = clean_label('label5')  # 気分 (mood)
posts['Y2_sound'] = clean_label('label2')  # 音  (sound)
posts['Y3_smell'] = clean_label('label3')  # 匂い (smell)

print(posts['Y1_mood'].value_counts())
print('\n', posts['Y2_sound'].value_counts())
print('\n', posts['Y3_smell'].value_counts())

Y1_mood
＃心地良い        7743
＃ワクワク        2315
その他          1820
＃驚き          1049
＃嬉しい          898
＃その他ポジティブ     833
＃退屈           655
Name: count, dtype: int64

 Y2_sound
＃車両          6476
＃無音          2883
＃風           2126
＃雑踏          1930
＃その他ポジティブ     784
＃生活音          648
その他           466
Name: count, dtype: int64

 Y3_smell
＃無臭          11342
＃花木           1386
＃食事系          1089
＃その他ポジティブ      762
＃その他ネガティブ      354
＃スイーツ系         302
その他             78
Name: count, dtype: int64


### Experiment A

In [4]:
"""
Experiment A: posting data WITHOUT image  ->  LightGBM
Features = coords + other tag columns (block A only).
Run:  DATA_ROOT=/path/to/extracted python3 exp_A_lgbm.py
"""
import os, warnings; warnings.filterwarnings("ignore")
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from common import load_master, build_tag_features, TARGETS

# use all CPU cores Slurm gave us (falls back to all local cores)
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 4))

# where to save results
OUT_DIR = os.environ.get("RESULT_DIR", "../result/exp_A/learn")
os.makedirs(OUT_DIR, exist_ok=True)

posts = load_master()

# save results table
posts_to_csv = os.path.join(OUT_DIR, "cleaned_posts.csv")
pd.DataFrame(posts).to_csv(posts_to_csv, index=False)
print(f"\nsaved -> {posts_to_csv}")


saved -> ../result/exp_A/learn/cleaned_posts.csv


In [ ]:
print(f"{'Target':10}{'N':>7}{'cls':>5}{'accuracy':>11}{'macroF1':>10}")

def build_tag_features(posts, target):
    """Feature block A: coords + other tag columns (excludes the target column)."""
    feat = posts[["lat", "lng"]].copy()
    exclude = TARGETS[target]  # don't feed the target's own source column
    tag_cols = [c for c in ["label1", "label2", "label3", "label4", "label5", "label6"] if c != exclude]
    for c in tag_cols:
        feat[c] = LabelEncoder().fit_transform(posts[c].astype(str))
    feat["source"] = LabelEncoder().fit_transform(posts["source"])
    return feat

rows = []
for t in TARGETS:
    print(f"\nTraining model for target: {t}")
    X = build_tag_features(posts, t)
    mask = posts[t].notna()
    X = X[mask.values]
    print(X.head())

    y = LabelEncoder().fit_transform(posts.loc[mask, t])
    print(y.shape)

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print(Xtr.shape, Xte.shape, ytr.shape, yte.shape)

    m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, n_jobs=N_JOBS, verbose=-1)
    m.fit(Xtr, ytr)
    p = m.predict(Xte)

    acc = accuracy_score(yte, p)
    print(f"Accuracy: {acc:.4f}")
    f1 = f1_score(yte, p, average="macro")
    print(f"Macro F1: {f1:.4f}")
    n = len(y); k = posts.loc[mask, t].nunique()
    print(posts.loc[mask, t].unique())
    print(f"{t:10}{n:>7}{k:>5}{acc:>11.3f}{f1:>10.3f}")
    rows.append({"target": t, "N": n, "classes": k, "accuracy": round(acc, 4), "macro_f1": round(f1, 4)})

# # save results table
# out_csv = os.path.join(OUT_DIR, "exp_A_results.csv")
# pd.DataFrame(rows).to_csv(out_csv, index=False)
# print(f"\nsaved -> {out_csv}")

Target          N  cls   accuracy   macroF1

Training model for target: Y1_mood
         lat         lng  label1  label2  label3  label4  label6  source
0  35.061717  135.749430       2      50      21      96       7       1
1  35.061154  135.749697      89     119      21      96       7       1
2  35.059234  135.750045      49     126      21      96       7       1
3  35.057362  135.750492      49     119      21      96       7       1
4  35.056920  135.751060       2      79      21      96       2       1
(15284,)
(12227, 8) (3057, 8) (12227,) (3057,)
Accuracy: 0.6297
Macro F1: 0.4567
['other' '＃その他ポジティブ' '＃ワクワク' '＃心地良い' '＃嬉しい' '＃退屈' '＃驚き']
Y1_mood     15284    7      0.630     0.457

Training model for target: Y2_sound
         lat         lng  label1  label3  label4  label5  label6  source
0  35.061717  135.749430       2      21      96       1       7       1
1  35.061154  135.749697      89      21      96       1       7       1
2  35.059234  135.750045      49      21    

### Experiment B

In [4]:
"""
Experiment B: posting data WITH image  ->  CLIP features + classifier
Extracts a 512-dim CLIP embedding per post image, then trains a simple
classifier for each target. Embeddings are cached to clip_emb.npz.

Install:  pip install torch torchvision ftfy regex tqdm
          pip install git+https://github.com/openai/CLIP.git
Run:      DATA_ROOT=/path/to/extracted python3 exp_B_clip.py

NOTE: ResNet/ViT variant = swap the encoder in embed_image() for a
torchvision model with its classification head removed; the rest is identical.
"""
import os, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, clip
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from common import load_master, find_image_path, TARGETS

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = os.environ.get("RESULT_DIR", "../result/exp_B/learn")
os.makedirs(OUT_DIR, exist_ok=True)
CACHE = os.path.join(OUT_DIR, "clip_emb.npz")

def build_embeddings(posts):
    """Return (embeddings [N,512], row index array) for posts with a real image."""
    if os.path.exists(CACHE):
        d = np.load(CACHE)
        return d["emb"], d["idx"]
    model, preprocess = clip.load("ViT-B/32", device=DEVICE)
    model.eval()
    embs, idxs = [], []
    for i, fn in tqdm(list(enumerate(posts["img_file"])), desc="CLIP"):
        p = find_image_path(fn)
        if p is None:
            continue
        try:
            img = preprocess(Image.open(p).convert("RGB")).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                v = model.encode_image(img).squeeze().cpu().numpy()
            embs.append(v); idxs.append(i)
        except Exception:
            continue
    emb = np.array(embs, dtype=np.float32); idx = np.array(idxs)
    np.savez(CACHE, emb=emb, idx=idx)
    return emb, idx

posts = load_master()
print(posts.head())
emb, idx = build_embeddings(posts)
print(f"images embedded: {len(idx)}")
print(f"{'Target':10}{'N':>7}{'accuracy':>11}{'macroF1':>10}")

# rows = []
# for t in TARGETS:
#     y_all = posts[t].values[idx]
#     keep = np.array([v is not None and str(v) != "nan" for v in y_all])
#     X = emb[keep]; y = LabelEncoder().fit_transform(y_all[keep].astype(str))
#     Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2,
#                                           random_state=42, stratify=y)
#     clf = LogisticRegression(max_iter=2000, C=1.0)
#     clf.fit(Xtr, ytr)
#     p = clf.predict(Xte)
#     acc = accuracy_score(yte, p); f1 = f1_score(yte, p, average="macro")
#     print(f"{t:10}{len(y):>7}{acc:>11.3f}{f1:>10.3f}")
#     rows.append({"target": t, "N": len(y),
#                  "accuracy": round(acc, 4), "macro_f1": round(f1, 4)})

# out_csv = os.path.join(OUT_DIR, "exp_B_results.csv")
# pd.DataFrame(rows).to_csv(out_csv, index=False)
# print(f"\nsaved -> {out_csv}")

  id                                         image_path        lat  \
0  3  https://delab2.kyoto-su.ac.jp/gnavi/app/backen...  35.061717   
1  4  https://delab2.kyoto-su.ac.jp/gnavi/app/backen...  35.061154   
2  5  https://delab2.kyoto-su.ac.jp/gnavi/app/backen...  35.059234   
3  6  https://delab2.kyoto-su.ac.jp/gnavi/app/backen...  35.057362   
4  7  https://delab2.kyoto-su.ac.jp/gnavi/app/backen...  35.056920   

          lng label1     label2 label3 label4     label5 content comment_en  \
0  135.749430   ＃その他        ＃車両    ＃無臭     ＃雨  ＃その他ネガティブ      なし       none   
1  135.749697    ＃風景      ＃雨(雪)    ＃無臭     ＃雨  ＃その他ネガティブ      なし       none   
2  135.750045    ＃看板  ＃雨(雪),＃車両    ＃無臭     ＃雨  ＃その他ネガティブ      なし       none   
3  135.750492    ＃看板      ＃雨(雪)    ＃無臭     ＃雨  ＃その他ネガティブ      なし       none   
4  135.751060   ＃その他  ＃車両,＃雨(雪)    ＃無臭     ＃雨  ＃その他ネガティブ      なし       none   

             timestamp label6    label7 source  \
0  2025-10-18 10:49:41   ＃その他  ＃難しい,＃簡単  quest   
1  2

CLIP: 100%|██████████| 15284/15284 [14:37<00:00, 17.43it/s] 


images embedded: 15283
Target          N   accuracy   macroF1


### Experiment C, A+C, B+C

In [1]:
import os, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from common import load_master, build_tag_features, build_spot_features, TARGETS

N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 4))
OUT_DIR = os.environ.get("RESULT_DIR", "../result/exp_C")
os.makedirs(OUT_DIR, exist_ok=True)
# CLIP embeddings produced by exp_B (needed only for B+C)
CLIP_CACHE = os.environ.get("CLIP_CACHE", "../result/exp_B/clip_emb.npz")

posts = load_master()
spot = build_spot_features(posts)              # block C, aligned to posts rows
rows = []

In [2]:
def evaluate(X, y, name, target):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2,
                                          random_state=42, stratify=y)
    m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05,
                           num_leaves=31, n_jobs=N_JOBS, verbose=-1)
    m.fit(Xtr, ytr); p = m.predict(Xte)
    acc = accuracy_score(yte, p); f1 = f1_score(yte, p, average="macro")
    print(f"  {name:6}{acc:>11.3f}{f1:>10.3f}")
    rows.append({"target": target, "experiment": name, "N": len(y),
                 "accuracy": round(acc, 4), "macro_f1": round(f1, 4)})

In [3]:
# optional CLIP embeddings for B+C
emb = idx = None
if os.path.exists(CLIP_CACHE):
    d = np.load(CLIP_CACHE); emb, idx = d["emb"], d["idx"]
else:
    print(f"(no CLIP cache at {CLIP_CACHE} -> skipping B+C; run exp_B first)")

for t in TARGETS:
    print(f"\n{t}   {'(exp)':6}{'accuracy':>11}{'macroF1':>10}")
    mask = posts[t].notna().values
    y = LabelEncoder().fit_transform(posts.loc[mask, t])

    # C only
    evaluate(spot[mask].values, y, "C", t)

    # A + C
    tag = build_tag_features(posts, t)[mask]
    ac = pd.concat([tag.reset_index(drop=True),
                    spot[mask].reset_index(drop=True)], axis=1)
    evaluate(ac.values, y, "A+C", t)

    # B + C  (only rows that have an image)
    if emb is not None:
        img_mask = np.zeros(len(posts), dtype=bool); img_mask[idx] = True
        both = mask & img_mask
        emb_row = {r: emb[k] for k, r in enumerate(idx)}
        idxs = np.where(both)[0]
        Xb = np.array([emb_row[r] for r in idxs])
        Xc = spot.values[idxs]
        Xbc = np.concatenate([Xb, Xc], axis=1)
        yb = LabelEncoder().fit_transform(posts.loc[both, t])
        evaluate(Xbc, yb, "B+C", t)

out_csv = os.path.join(OUT_DIR, "exp_C_results_n.csv")
pd.DataFrame(rows).to_csv(out_csv, index=False)
print(f"\nsaved -> {out_csv}")


Y1_mood   (exp)    accuracy   macroF1
  C           0.512     0.204
  A+C         0.630     0.456
  B+C         0.535     0.197

Y2_sound   (exp)    accuracy   macroF1
  C           0.520     0.316
  A+C         0.710     0.618
  B+C         0.558     0.332

Y3_smell   (exp)    accuracy   macroF1
  C           0.737     0.215
  A+C         0.830     0.492
  B+C         0.760     0.198

saved -> ../result/exp_C/exp_C_results_n.csv
